# 05 Analytics

> Change only `RAW` / `PROCESSED` paths if your folder location is different.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, json

BASE = Path(r"C:\CHANGE\THIS\TO\YOUR\PROJECT")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
arrivals = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_arrivals.csv")
master = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_master.csv")
transport = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_transport_logistics.csv")
master
with open(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_price_and_msp.json", encoding="utf-8") as f:
    prices = pd.DataFrame(json.load(f))

weather = pd.read_excel(RAW / "C:\\Users\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_weather_sensors.xlsx")

## Core KPIs

In [4]:
# ==========================================
# 05 ANALYTICS - LOAD DATA + CREATE KPIs
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np

P = Path("../data/processed")

# Load data-model tables
arrivals = pd.read_csv(P / "arrivals_fact.csv")
PR = pd.read_csv(P / "prices_fact.csv")
transport = pd.read_csv(P / "transport_fact.csv")

# ------------------------------------------------
# SAFETY: CREATE ARRIVAL QUANTITY IN QTL IF NEEDED
# ------------------------------------------------

if "arrival_quantity_qtl" not in arrivals.columns:

    if "arrival_quantity" in arrivals.columns:

        arrivals["arrival_quantity"] = pd.to_numeric(
            arrivals["arrival_quantity"],
            errors="coerce"
        )

        if "unit" in arrivals.columns:

            arrivals["unit"] = (
                arrivals["unit"]
                .astype(str)
                .str.strip()
                .str.lower()
            )

            unit_map = {
                "kg": 0.01,
                "kgs": 0.01,
                "kilo": 0.01,
                "q": 1,
                "qtl": 1,
                "quintal": 1,
                "quintals": 1,
                "mt": 10,
                "t": 10,
                "tonne": 10,
                "tonnes": 10
            }

            arrivals["quantity_to_qtl_factor"] = (
                arrivals["unit"].map(unit_map)
            )

            arrivals["arrival_quantity_qtl"] = (
                arrivals["arrival_quantity"]
                * arrivals["quantity_to_qtl_factor"]
            )

        else:
            arrivals["arrival_quantity_qtl"] = np.nan

    else:
        arrivals["arrival_quantity_qtl"] = np.nan


# ------------------------------------------------
# CLEAN PRICE COLUMNS
# ------------------------------------------------

for col in ["modal_price", "msp"]:
    if col in PR.columns:
        PR[col] = (
            PR[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("₹", "", regex=False)
            .str.replace("Rs.", "", regex=False)
            .str.replace("Rs", "", regex=False)
            .str.replace("INR", "", regex=False)
            .str.strip()
        )

        PR[col] = pd.to_numeric(PR[col], errors="coerce")


# ------------------------------------------------
# KPI 1: TOTAL ARRIVALS
# ------------------------------------------------

total_arrivals_qtl = arrivals["arrival_quantity_qtl"].sum()


# ------------------------------------------------
# KPI 2: AVERAGE MODAL PRICE
# ------------------------------------------------

avg_modal_price = PR["modal_price"].mean()


# ------------------------------------------------
# KPI 3: MSP GAP
# ------------------------------------------------

PR["msp_gap_pct"] = np.where(
    PR["msp"] > 0,
    ((PR["modal_price"] - PR["msp"]) / PR["msp"]) * 100,
    np.nan
)


# ------------------------------------------------
# KPI 4: PRICE CRASH RATE
# ------------------------------------------------

if "date" in PR.columns:

    PR["date"] = pd.to_datetime(
        PR["date"],
        errors="coerce"
    )

    PR = PR.sort_values("date")

    PR["price_change_pct"] = (
        PR.groupby(["mandi_id", "crop_name"])["modal_price"]
        .pct_change() * 100
    )

    price_crash_rate = (
        PR["price_change_pct"].le(-10).mean() * 100
    )

else:
    price_crash_rate = np.nan


# ------------------------------------------------
# KPI 5: TRANSPORT
# ------------------------------------------------

if "transit_hours" in transport.columns:

    transport["transit_hours"] = pd.to_numeric(
        transport["transit_hours"],
        errors="coerce"
    )

    avg_transit_hours = transport["transit_hours"].mean()

    transit_delay_rate = (
        transport["transit_hours"].gt(24).mean() * 100
    )

else:

    avg_transit_hours = np.nan
    transit_delay_rate = np.nan


# ------------------------------------------------
# DISPLAY KPIs
# ------------------------------------------------

print("========================================")
print("AGRI MANDI ANALYTICS")
print("========================================")

print(f"Total Arrivals: {total_arrivals_qtl:,.2f} Qtl")
print(f"Average Modal Price: ₹{avg_modal_price:,.2f}")
print(f"Price Crash Rate: {price_crash_rate:.2f}%")
print(f"Average Transit Time: {avg_transit_hours:.2f} hours")
print(f"Transit Delay Rate: {transit_delay_rate:.2f}%")

print("\nArrival columns:")
print(arrivals.columns.tolist())

print("\nPrice columns:")
print(PR.columns.tolist())

AGRI MANDI ANALYTICS
Total Arrivals: 4,800,534.09 Qtl
Average Modal Price: ₹3,804.70
Price Crash Rate: 3.98%
Average Transit Time: 12.94 hours
Transit Delay Rate: 0.00%

Arrival columns:
['arrival_id', 'date', 'mandi_id', 'crop_name', 'variety', 'arrival_quantity_qtl', 'farmer_count']

Price columns:
['record_id', 'date', 'mandi_id', 'district', 'crop_name', 'min_price', 'max_price', 'modal_price', 'msp', 'msp_gap_pct', 'price_change_pct']


## KPI tables for dashboard

In [11]:
# ==========================================
# 05 ANALYTICS - COMPLETE KPI CALCULATION
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np

# Output path
P = Path("../data/processed")
out = P / "analytics"
out.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# LOAD DATA
# ------------------------------------------------

arrivals = pd.read_csv(P / "arrivals_fact.csv")
PR = pd.read_csv(P / "prices_fact.csv")
transport = pd.read_csv(P / "transport_fact.csv")

# ------------------------------------------------
# ARRIVALS
# ------------------------------------------------

if "arrival_quantity_qtl" not in arrivals.columns:

    arrivals["arrival_quantity"] = pd.to_numeric(
        arrivals["arrival_quantity"],
        errors="coerce"
    )

    arrivals["unit"] = (
        arrivals["unit"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    unit_map = {
        "kg": 0.01,
        "kgs": 0.01,
        "kilo": 0.01,
        "q": 1,
        "qtl": 1,
        "quintal": 1,
        "quintals": 1,
        "mt": 10,
        "t": 10,
        "tonne": 10,
        "tonnes": 10
    }

    arrivals["arrival_quantity_qtl"] = (
        arrivals["arrival_quantity"]
        * arrivals["unit"].map(unit_map)
    )

# Date
arrivals["date"] = pd.to_datetime(
    arrivals["date"],
    errors="coerce"
)

# ------------------------------------------------
# PRICES
# ------------------------------------------------

for col in ["modal_price", "msp"]:

    PR[col] = (
        PR[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("₹", "", regex=False)
        .str.replace("Rs.", "", regex=False)
        .str.replace("Rs", "", regex=False)
        .str.replace("INR", "", regex=False)
        .str.strip()
    )

    PR[col] = pd.to_numeric(
        PR[col],
        errors="coerce"
    )

PR["date"] = pd.to_datetime(
    PR["date"],
    errors="coerce"
)

# ------------------------------------------------
# KPI CALCULATIONS
# ------------------------------------------------

total_arrivals_qtl = arrivals[
    "arrival_quantity_qtl"
].sum()

avg_modal_price = PR[
    "modal_price"
].mean()

msp_gap_pct = np.where(
    PR["msp"] > 0,
    ((PR["modal_price"] - PR["msp"]) / PR["msp"]) * 100,
    np.nan
)

PR["msp_gap_pct"] = msp_gap_pct

# Price crash
PR = PR.sort_values("date")

PR["price_change_pct"] = (
    PR.groupby(
        ["mandi_id", "crop_name"]
    )["modal_price"]
    .pct_change() * 100
)

price_crash_rate = (
    PR["price_change_pct"]
    .le(-10)
    .mean() * 100
)

# Transport
transport["transit_hours"] = pd.to_numeric(
    transport["transit_hours"],
    errors="coerce"
)

avg_transit_hours = transport[
    "transit_hours"
].mean()

transit_delay_rate = (
    transport["transit_hours"]
    .gt(24)
    .mean() * 100
)

# ------------------------------------------------
# CREATE KPI TABLE
# ------------------------------------------------

kpi = pd.DataFrame({
    "metric": [
        "Total Arrivals",
        "Average Modal Price",
        "Price Crash Rate",
        "Average Transit Hours",
        "Transit Delay Rate"
    ],
    "value": [
        total_arrivals_qtl,
        avg_modal_price,
        price_crash_rate,
        avg_transit_hours,
        transit_delay_rate
    ],
    "unit": [
        "Qtl",
        "INR",
        "%",
        "Hours",
        "%"
    ]
})

# ------------------------------------------------
# MANDI KPI
# ------------------------------------------------

mandi_kpi = (
    PR.groupby("mandi_id", as_index=False)
    .agg(
        avg_modal_price=("modal_price", "mean"),
        avg_msp=("msp", "mean"),
        avg_msp_gap_pct=("msp_gap_pct", "mean")
    )
)

# ------------------------------------------------
# CROP KPI
# ------------------------------------------------

crop_kpi = (
    arrivals.groupby("crop_name", as_index=False)
    .agg(
        total_arrival_qtl=(
            "arrival_quantity_qtl",
            "sum"
        ),
        avg_arrival_qtl=(
            "arrival_quantity_qtl",
            "mean"
        )
    )
)

# ------------------------------------------------
# DAILY ARRIVALS
# ------------------------------------------------

daily_arrivals = (
    arrivals
    .groupby("date", as_index=False)
    .agg(
        arrival_quantity_qtl=(
            "arrival_quantity_qtl",
            "sum"
        )
    )
    .sort_values("date")
)

daily_arrivals["rolling_7d"] = (
    daily_arrivals[
        "arrival_quantity_qtl"
    ]
    .rolling(
        7,
        min_periods=3
    )
    .mean()
)

# ------------------------------------------------
# SAVE ANALYTICS
# ------------------------------------------------

kpi.to_csv(
    out / "kpis.csv",
    index=False
)

mandi_kpi.to_csv(
    out / "mandi_kpi.csv",
    index=False
)

crop_kpi.to_csv(
    out / "crop_kpi.csv",
    index=False
)

daily_arrivals.to_csv(
    out / "daily_arrivals.csv",
    index=False
)

PR.to_csv(
    out / "prices_analytics.csv",
    index=False
)

# ------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------

print("========================================")
print("ANALYTICS COMPLETED SUCCESSFULLY")
print("========================================")

display(kpi)

print("\nMandi KPI:")
display(mandi_kpi.head())

print("\nCrop KPI:")
display(crop_kpi)

print("\nDaily Arrivals:")
display(daily_arrivals.head())

ANALYTICS COMPLETED SUCCESSFULLY


,metric,value,unit
0,Total Arrivals,4.800534e+06,Qtl
1,Average Modal Price,3.804704e+03,INR
2,Price Crash Rate,3.983333e+00,%
3,Average Transit Hours,1.294199e+01,Hours
4,Transit Delay Rate,0.000000e+00,%



Mandi KPI:


,mandi_id,avg_modal_price,avg_msp,avg_msp_gap_pct
0,001,2699.456200,2691.200000,-0.154066
1,002,4119.495608,3719.666667,4.228276
2,003,3536.114610,3403.200000,2.681165
3,004,3001.413050,3167.875000,0.522390
4,005,3089.585000,3841.777778,-3.543627



Crop KPI:


,crop_name,total_arrival_qtl,avg_arrival_qtl
0,Basmati,104087.9950,239.282747
1,Chawal,113593.4107,245.873183
2,Corn,142972.9450,235.928952
3,Cotton,168739.3032,229.890059
4,Dhaan,84481.9650,227.714191
5,GEHUN,108300.7193,233.910841
6,Ganna,157588.0927,241.329392
7,Ganne,150205.0370,234.329231
8,Gehun,111080.1890,215.689687
9,Kanak,110736.6856,222.362822



Daily Arrivals:


,date,arrival_quantity_qtl,rolling_7d
0,2026-01-01,2244.2700,NaN
1,2026-01-02,2960.0700,NaN
2,2026-01-03,2093.6400,2432.660000
3,2026-01-04,3539.6525,2709.408125
4,2026-01-05,2400.0800,2647.542500
